# Chunked cosine search: validate scores and document IDs

This notebook checks a retrieval algorithm against an independent dense reference. It uses synthetic vectors, planted nearest neighbors, uneven chunks, and deliberate corruptions that the validator must reject.

**Disclosure:** I work on CloudGPU. This is a CPU correctness exercise, not a GPU benchmark, service endorsement, or measurement of model quality. AI assisted the preparation of this notebook. All code cells were executed locally on the box; the host platform must run them again before publication.

No account key, model download, external dataset, or network call is required. Use a CPU runtime with NumPy. The example prints the actual environment rather than assuming a package version. Do not enable Python's optimization flag: the validator uses assertions.

Related article: https://dev.to/cloudgpu_app/build-and-verify-chunked-cosine-search-with-python-and-pytorch-on-a-rented-gpu-5ep1

The article uses PyTorch; this smaller NumPy exercise is a distinct CPU-only adaptation. It does not test the article's CUDA path.


In [ ]:
import json
import platform
import numpy as np

SEED = 20261111
N_ROWS, DIM, N_QUERIES, K = 2053, 64, 16, 5
rng = np.random.default_rng(SEED)
corpus = rng.normal(size=(N_ROWS, DIM)).astype(np.float32)
corpus /= np.linalg.norm(corpus, axis=1, keepdims=True)
expected_first_ids = np.arange(N_QUERIES) * 127
queries = corpus[expected_first_ids].copy()
print(json.dumps({"python": platform.python_version(), "numpy": np.__version__, "device": "cpu", "seed": SEED}))


## Keep global row IDs when merging block candidates

For unit-normalized vectors, the dot product is cosine similarity. A discarded candidate already has at least `k` candidates scoring no lower in its own block; keeping the current top `k` is sufficient when merging blocks. Exact ties may produce different valid IDs; the validator compares scores and checks the score of each returned ID.

`chunk_rows=257` leaves an uneven tail. A chunk size smaller than `k` exercises the early merge path.


In [ ]:
def chunked_topk(corpus, queries, k, chunk_rows):
    if (corpus.ndim != 2 or queries.ndim != 2
            or corpus.shape[1] != queries.shape[1]):
        raise ValueError("Expected two compatible 2D arrays")
    if not 1 <= k <= len(corpus) or chunk_rows < 1:
        raise ValueError("Invalid k or chunk size")
    if not np.isfinite(corpus).all() or not np.isfinite(queries).all():
        raise ValueError("Inputs must be finite")
    scores = np.empty((len(queries), 0), dtype=corpus.dtype)
    ids = np.empty((len(queries), 0), dtype=np.int64)
    for offset in range(0, len(corpus), chunk_rows):
        block = corpus[offset:offset + chunk_rows]
        similarity = queries @ block.T
        local_k = min(k, len(block))
        positions = np.argsort(-similarity, axis=1, kind="stable")[:, :local_k]
        local_scores = np.take_along_axis(similarity, positions, axis=1)
        candidates = np.concatenate([scores, local_scores], axis=1)
        candidate_ids = np.concatenate([ids, positions + offset], axis=1)
        keep = np.argsort(-candidates, axis=1, kind="stable")[:, :min(k, candidates.shape[1])]
        scores = np.take_along_axis(candidates, keep, axis=1)
        ids = np.take_along_axis(candidate_ids, keep, axis=1)
    return scores, ids


def validate(corpus, queries, expected_first_ids, scores, ids, k):
    assert scores.shape == ids.shape == (len(queries), k)
    assert np.isfinite(scores).all()
    assert np.issubdtype(ids.dtype, np.integer)
    assert ((ids >= 0) & (ids < len(corpus))).all()
    assert (scores[:, :-1] >= scores[:, 1:]).all()
    assert all(len(np.unique(row)) == k for row in ids)
    assert np.array_equal(ids[:, 0], expected_first_ids)
    reference = queries.astype(np.float64) @ corpus.astype(np.float64).T
    expected_scores = np.sort(reference, axis=1)[:, -k:][:, ::-1]
    selected_scores = np.take_along_axis(reference, ids, axis=1)
    np.testing.assert_allclose(scores, expected_scores, rtol=1e-4, atol=1e-6)
    np.testing.assert_allclose(scores, selected_scores, rtol=1e-4, atol=1e-6)
    return float(np.max(np.abs(scores - expected_scores)))


In [ ]:
correct_runs = []
for chunk_rows in [257, 3, N_ROWS + 1]:
    scores, ids = chunked_topk(corpus, queries, K, chunk_rows)
    error = validate(corpus, queries, expected_first_ids, scores, ids, K)
    correct_runs.append({"chunk_rows": chunk_rows, "assertions_passed": True,
                         "max_score_error": error})
print(json.dumps(correct_runs, indent=2))


## Make the checker fail on purpose

A successful process exit is not enough. Corrupt one property at a time: global IDs, score-to-ID correspondence, ordering, or finiteness. A validator that accepts a corrupted fixture is not an adequate regression test.


In [ ]:
scores, ids = chunked_topk(corpus, queries, K, 257)
bad_ids = ids.copy()
bad_ids[:, 1] = (bad_ids[:, 1] + 1) % len(corpus)
no_offsets = ids % 257
reversed_scores = scores[:, ::-1].copy()
nan_scores = scores.copy()
nan_scores[0, 0] = np.nan
corruptions = [("score_id_mismatch", scores, bad_ids),
               ("missing_chunk_offsets", scores, no_offsets),
               ("reversed_scores", reversed_scores, ids),
               ("non_finite_score", nan_scores, ids)]
rejected = []
for name, candidate_scores, candidate_ids in corruptions:
    try:
        validate(corpus, queries, expected_first_ids, candidate_scores, candidate_ids, K)
    except AssertionError:
        rejected.append(name)
    else:
        raise AssertionError("Validator accepted corruption: " + name)
for name, k, chunk in [("zero_chunk", K, 0), ("oversized_k", N_ROWS + 1, 257)]:
    try:
        chunked_topk(corpus, queries, k, chunk)
    except ValueError:
        rejected.append(name)
    else:
        raise AssertionError("Invalid input was accepted: " + name)
print(json.dumps({"rejected_cases": rejected, "count": len(rejected)}, indent=2))


## What this does and does not establish

- It checks indexing, score ordering, numerical agreement, and several error paths on one small synthetic fixture.
- It is not a semantic search evaluation: there is no encoder, relevance judgment, or language model.
- It is not a GPU execution transcript or a throughput comparison. No hardware-performance claim follows from these results.
- The dense reference deliberately fits this small example. Do not allocate it for an unrestricted corpus.
- For real data, validate normalization, reject zero vectors, define tie policy, preserve document IDs, and use a representative reference sample.

You can run the same CPU notebook in JupyterLab, Kaggle, or Colab. Export the notebook and its outputs with the environment record when comparing runs. Do not upload private vectors or production credentials to a public notebook.

Maintainer: yang@cloudgpu.app
